In [1]:
import requests

# Test that Ollama is running
response = requests.get("http://localhost:11434/api/tags")

if response.status_code == 200:
    models = response.json()["models"]
    print("✅ Ollama is running!")
    print("Available models:")
    for model in models:
        print(f"  - {model['name']}")
else:
    print("❌ Ollama is not running. Start it first!")

✅ Ollama is running!
Available models:
  - qwen2.5vl:3b


In [2]:
import base64
import json

# Load and encode the invoice image
# First convert PDF to image
import fitz

pdf_path = "../invoices/facture_test.pdf"
doc = fitz.open(pdf_path)
page = doc[0]
pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
pix.save("../outputs/facture_test.png")
print("✅ PDF converted to image successfully!")

✅ PDF converted to image successfully!


In [1]:
import base64
import requests

# Load and encode the image
with open("../outputs/facture_test.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

# Prompt with currency added
prompt = """You are an invoice data extraction assistant.
Extract the following fields from this invoice image and return ONLY a valid JSON object, nothing else. No explanation, no markdown, just the raw JSON.

{
  "invoice_number": "",
  "date": "",
  "supplier": "",
  "client": "",
  "currency": "",
  "montant_ht": 0,
  "tva": 0,
  "montant_ttc": 0,
  "line_items": [
    {"description": "", "quantity": 0, "unit_price": 0, "total": 0}
  ]
}"""

print("⏳ Sending invoice to Vision LLM... please wait")
response = requests.post("http://localhost:11434/api/generate", json={
    "model": "qwen2.5vl:7b",
    "prompt": prompt,
    "images": [image_b64],
    "stream": False
})

result = response.json()["response"]
print("✅ Response received!")
print(result)

⏳ Sending invoice to Vision LLM... please wait
✅ Response received!
```json
{
  "invoice_number": "FAC-2026-00142",
  "date": "20/06/2026",
  "supplier": "ATLAS TECH SOLUTIONS SARL",
  "client": "MAROC DISTRIBUTION SARL",
  "currency": "MAD",
  "montant_ht": 45300,
  "tva": 9060,
  "montant_ttc": 54360,
  "line_items": [
    {"description": "Développement application web (React/Node.js)", "quantity": 1, "unit_price": 15000, "total": 15000},
    {"description": "Intégration API REST et base de données", "quantity": 1, "unit_price": 8500, "total": 8500},
    {"description": "Formation et documentation technique (5 jours)", "quantity": 5, "unit_price": 2000, "total": 10000},
    {"description": "Maintenance et support mensuel", "quantity": 2, "unit_price": 3500, "total": 7000},
    {"description": "Hébergement serveur cloud (6 mois)", "quantity": 6, "unit_price": 800, "total": 4800}
  ]
}
```


In [2]:
import json
import re

# Clean the response (remove markdown code blocks if present)
clean = re.sub(r'```json|```', '', result).strip()

try:
    data = json.loads(clean)
    print("✅ JSON parsed successfully!")
    print("\n📋 Extracted Data:")
    print(f"  Invoice Number : {data['invoice_number']}")
    print(f"  Date           : {data['date']}")
    print(f"  Supplier       : {data['supplier']}")
    print(f"  Client         : {data['client']}")
    print(f"  Montant HT     : {data['montant_ht']} MAD")
    print(f"  TVA            : {data['tva']} MAD")
    print(f"  Montant TTC    : {data['montant_ttc']} MAD")

    print("\n🔍 Validation:")
    calculated_ttc = data['montant_ht'] + data['tva']
    expected_ttc = data['montant_ttc']
    difference = abs(calculated_ttc - expected_ttc)

    if difference <= 1:
        print(f"  ✅ HT + TVA = TTC → {data['montant_ht']} + {data['tva']} = {calculated_ttc} ✅")
    else:
        print(f"  ❌ HALLUCINATION DETECTED!")
        print(f"  HT + TVA = {calculated_ttc} but TTC = {expected_ttc}")
        print(f"  Difference: {difference} MAD")

except json.JSONDecodeError as e:
    print(f"❌ JSON parsing failed: {e}")

✅ JSON parsed successfully!

📋 Extracted Data:
  Invoice Number : FAC-2026-00142
  Date           : 20/06/2026
  Supplier       : ATLAS TECH SOLUTIONS SARL
  Client         : MAROC DISTRIBUTION SARL
  Montant HT     : 45300 MAD
  TVA            : 9060 MAD
  Montant TTC    : 54360 MAD

🔍 Validation:
  ✅ HT + TVA = TTC → 45300 + 9060 = 54360 ✅


In [14]:
# Test on real invoice
import fitz
import base64
import requests

# Change this to your real invoice filename
pdf_path = "../invoices/invoice.pdf"  

# Convert to image
doc = fitz.open(pdf_path)
page = doc[0]
pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
pix.save("../outputs/real_invoice_test.png")
print("✅ Real invoice converted to image!")

# Encode image
with open("../outputs/real_invoice_test.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

# Prompt
prompt = """You are an invoice data extraction assistant.
Extract the following fields from this invoice image and return ONLY a valid JSON object, nothing else. No explanation, no markdown, just the raw JSON.

{
  "invoice_number": "",
  "date": "",
  "supplier": "",
  "client": "",
  "currency": "",
  "montant_ht": 0,
  "tva": 0,
  "montant_ttc": 0,
  "line_items": [
    {"description": "", "quantity": 0, "unit_price": 0, "total": 0}
  ]
}"""

print("⏳ Sending real invoice to Vision LLM... please wait")
response = requests.post("http://localhost:11434/api/generate", json={
    "model": "qwen2.5vl:7b",
    "prompt": prompt,
    "images": [image_b64],
    "stream": False,
    "keep_alive": "30m"
})

result_real = response.json()["response"]
print("✅ Response received!")
print(result_real)

✅ Real invoice converted to image!
⏳ Sending real invoice to Vision LLM... please wait
✅ Response received!
```json
{
  "invoice_number": "F20250511131",
  "date": "dimanche 11 mai 2025",
  "supplier": "Damanesign SA",
  "client": "BLUZELLE CENTER",
  "currency": "DH",
  "montant_ht": 800,
  "tva": 160.0,
  "montant_ttc": 960.0,
  "line_items": [
    {
      "description": "Pack de 100 signatures simples Pack de 100 signatures",
      "quantity": 1,
      "unit_price": 800,
      "total": 800
    }
  ]
}
```


In [6]:
import json
import re

# Clean the response
clean_real = re.sub(r'```json|```', '', result_real).strip()

try:
    data_real = json.loads(clean_real)
    
    # Convert to numbers in case model returns strings
    data_real['montant_ht'] = float(str(data_real['montant_ht']).replace(',', '.').replace(' ', ''))
    data_real['tva'] = float(str(data_real['tva']).replace(',', '.').replace(' ', ''))
    data_real['montant_ttc'] = float(str(data_real['montant_ttc']).replace(',', '.').replace(' ', ''))
    
    # Get currency
    currency = data_real.get('currency', 'N/A')
    
    print("✅ JSON parsed successfully!")
    print("\n📋 Extracted Data:")
    print(f"  Invoice Number : {data_real['invoice_number']}")
    print(f"  Date           : {data_real['date']}")
    print(f"  Supplier       : {data_real['supplier']}")
    print(f"  Client         : {data_real['client']}")
    print(f"  Currency       : {currency}")
    print(f"  Montant HT     : {data_real['montant_ht']} {currency}")
    print(f"  TVA            : {data_real['tva']} {currency}")
    print(f"  Montant TTC    : {data_real['montant_ttc']} {currency}")

    print("\n🔍 Validation:")
    calculated_ttc = data_real['montant_ht'] + data_real['tva']
    expected_ttc = data_real['montant_ttc']
    difference = abs(calculated_ttc - expected_ttc)

    if difference <= 1:
        print(f"  ✅ HT + TVA = TTC → {data_real['montant_ht']} + {data_real['tva']} = {calculated_ttc} {currency} ✅")
    else:
        print(f"  ❌ HALLUCINATION DETECTED!")
        print(f"  HT + TVA = {calculated_ttc} {currency} but TTC = {expected_ttc} {currency}")
        print(f"  Difference: {difference} {currency}")

except json.JSONDecodeError as e:
    print(f"❌ JSON parsing failed: {e}")

✅ JSON parsed successfully!

📋 Extracted Data:
  Invoice Number : XHVVGSLW-0076
  Date           : May 19, 2026
  Supplier       : Anthropic, PBC
  Client         : Générafi mers sultan
  Currency       : USD
  Montant HT     : 25.0 USD
  TVA            : 0.0 USD
  Montant TTC    : 25.0 USD

🔍 Validation:
  ✅ HT + TVA = TTC → 25.0 + 0.0 = 25.0 USD ✅


In [5]:
# --- MiniCPM-V speed/accuracy test on the same real invoice ---
import time
import base64
import requests
import json
import re

# Reuse the same image already converted earlier
with open("../invoices/facture_test_wrong_math.png", "rb") as f:
    image_b64_mini = base64.b64encode(f.read()).decode()

prompt_mini = """You are an invoice data extraction assistant.
Extract the following fields from this invoice image and return ONLY a valid JSON object, nothing else. No explanation, no markdown, just the raw JSON.

{
  "invoice_number": "",
  "date": "",
  "supplier": "",
  "client": "",
  "currency": "",
  "montant_ht": 0,
  "tva": 0,
  "montant_ttc": 0,
  "line_items": [
    {"description": "", "quantity": 0, "unit_price": 0, "total": 0}
  ]
}"""

print("⏳ Sending real invoice to MiniCPM-V... please wait")
start_mini = time.time()

response_mini = requests.post("http://localhost:11434/api/generate", json={
    "model": "minicpm-v",
    "prompt": prompt_mini,
    "images": [image_b64_mini],
    "stream": False
})

end_mini = time.time()
result_mini = response_mini.json()["response"]

print(f"⏱ Time taken: {end_mini - start_mini:.2f} seconds")
print("✅ Response received!")
print(result_mini)

# --- Validation (same logic as before) ---
clean_mini = re.sub(r'```json|```', '', result_mini).strip()

try:
    data_mini = json.loads(clean_mini)

    data_mini['montant_ht'] = float(str(data_mini['montant_ht']).replace(',', '.').replace(' ', ''))
    data_mini['tva'] = float(str(data_mini['tva']).replace(',', '.').replace(' ', ''))
    data_mini['montant_ttc'] = float(str(data_mini['montant_ttc']).replace(',', '.').replace(' ', ''))

    currency_mini = data_mini.get('currency', 'N/A')

    print("✅ JSON parsed successfully!")
    print("\n📋 Extracted Data (MiniCPM-V):")
    print(f"  Invoice Number : {data_mini['invoice_number']}")
    print(f"  Date           : {data_mini['date']}")
    print(f"  Supplier       : {data_mini['supplier']}")
    print(f"  Client         : {data_mini['client']}")
    print(f"  Currency       : {currency_mini}")
    print(f"  Montant HT     : {data_mini['montant_ht']} {currency_mini}")
    print(f"  TVA            : {data_mini['tva']} {currency_mini}")
    print(f"  Montant TTC    : {data_mini['montant_ttc']} {currency_mini}")

    print("\n🔍 Validation:")
    calculated_ttc_mini = data_mini['montant_ht'] + data_mini['tva']
    expected_ttc_mini = data_mini['montant_ttc']
    difference_mini = abs(calculated_ttc_mini - expected_ttc_mini)

    if difference_mini <= 1:
        print(f"  ✅ HT + TVA = TTC → {data_mini['montant_ht']} + {data_mini['tva']} = {calculated_ttc_mini} {currency_mini} ✅")
    else:
        print(f"  ❌ HALLUCINATION DETECTED!")
        print(f"  HT + TVA = {calculated_ttc_mini} {currency_mini} but TTC = {expected_ttc_mini} {currency_mini}")
        print(f"  Difference: {difference_mini} {currency_mini}")

except json.JSONDecodeError as e:
    print(f"❌ JSON parsing failed: {e}")

⏳ Sending real invoice to MiniCPM-V... please wait
⏱ Time taken: 57.61 seconds
✅ Response received!
{
  "invoice_number": "",
  "date": "25/06/2026",
  "supplier": "NOVA DIGITAL SERVICES SARL",
  "client": "GREENFIELD AGRO S.A.R.L.",
  "currency": "MAD",
  "montant_ht": {
    "Qte": [
      {"Description": "Conception UX/UI platforme mobile web", "Prix Unitaire (MAD)": "12 000.00", "Total HT (MAD)": "12 000.00"},
      {"Description": "Developpement application plateforme mobile (Flutter)", "Prix Unitaire (MAD)": "22 000.00", "Total HT (MAD)": "22 000.00"},
      {"Description": "Tests et assurance qualite", "Prix Unitaire (MAD)": "1 500.00", "Total HT (MAD)": "4 500.00"},
      {"Description": "Formation utilisateurs (2 jours)", "Prix Unitaire (MAD)": "1 800.00", "Total HT (MAD)": "3 600.00"},
      {"Description": "Support technique (3 mois)", "Prix Unitaire (MAD)": "900.00", "Total HT (MAD)": "2 700.00"}
    ]
  },
  "montant_ttc": {
    "Sous-total HT": [
      {"Montant": "44 800.

ValueError: could not convert string to float: "{'Qte':[{'Description':'ConceptionUX/UIplatformemobileweb'.'PrixUnitaire(MAD)':'12000.00'.'TotalHT(MAD)':'12000.00'}.{'Description':'Developpementapplicationplateformemobile(Flutter)'.'PrixUnitaire(MAD)':'22000.00'.'TotalHT(MAD)':'22000.00'}.{'Description':'Testsetassurancequalite'.'PrixUnitaire(MAD)':'1500.00'.'TotalHT(MAD)':'4500.00'}.{'Description':'Formationutilisateurs(2jours)'.'PrixUnitaire(MAD)':'1800.00'.'TotalHT(MAD)':'3600.00'}.{'Description':'Supporttechnique(3mois)'.'PrixUnitaire(MAD)':'900.00'.'TotalHT(MAD)':'2700.00'}]}"